# ViT Unfolded-Attention CRP Walkthrough

End-to-end concept attribution + relevance maximisation on a **small
ViT** (default `vit_small`; `vit_base` and DINOv3 ViT-L also supported)
with the unfolded-attention refactor and the AlphaBeta (0.5, 0.5)
bilinear rule, using the three concept classes (`HeadConcept`,
`EmbeddingDimConcept`, `TokenConcept`) hookable at any of the
`LRPInspectionLayer` sites (`q_lrp_probe`, `k_lrp_probe`,
`v_lrp_probe`) or at `proj_drop`.

## Setup — what to run before opening the notebook

**1. Install the environment.** From the repo root:

```bash
uv sync
```

All dependencies (torch, timm, lightning, jupyter, huggingface-hub,
etc.) are declared in `pyproject.toml` and installed into `.venv/`.

**2. Pick a (base, head, train-ds).** Defaults in section 2 are
`vit_small + linear + funny-birds-train-clean` — a small, fast ViT
that fine-tunes in ~1 h on one GPU and runs CRP comfortably. Change
them in section 2. Available choices:

| | options |
|---|---|
| **base**    | `vit_small` *(timm vit_small_patch16_224, **default**, 22 M)* &nbsp;·&nbsp; `vit_base` *(vit_base_patch16_224, 86 M)* &nbsp;·&nbsp; `vit_dinov3` *(vit_large_patch16_dinov3, 304 M; Eva + LayerScale + RoPE)* |
| **head**    | `linear` *(cls-token classifier, default)* &nbsp;·&nbsp; `attentive` *(learned-query attention pool)* &nbsp;·&nbsp; `block` *(one transformer block)* |
| **train-ds** | `funny-birds-train-clean` *(50 birds + GT part maps, default)* &nbsp;·&nbsp; `dsprites` &nbsp;·&nbsp; `colored-mnist-train` &nbsp;·&nbsp; `imagenette-train` &nbsp;·&nbsp; `imagenet-val-hf` |

All datasets **auto-download** on first use — no manual setup.

**3. Train the probe.** The default `vit_small` is a full-backbone
OneCycle fine-tune (the validated baseline recipe):

```bash
uv run python -m experiments.train_probe finetune --from-scratch \
  --base vit_small --head linear --train-ds funny-birds-train-clean \
  --epochs 25 --patience 25 --backbone-lr 5e-4 --head-lr 5e-3 \
  --weight-decay 0.05 --batch-size 64 --accumulate-grad-batches 2 \
  --layerwise-lr-decay 0.7 --scheduler onecycle --onecycle-pct-start 0.1 \
  --randaugment --label-smoothing 0.1 --val-frac 0.1 --num-workers 3 --seed 0
```

Output lands at `data/runs/finetune_vit_small_funny-birds-train-clean/<ts>/best.pt`,
auto-resolved by section 2. For a **frozen-backbone probe** (the
DINOv3 style — cheap, train only the head) use instead:

```bash
uv run train-probe cache  <base> <dataset> --kind <cls|tokens>
uv run train-probe train  <base> <head>    <dataset>
```

The probe-loading cell (section 2) raises `FileNotFoundError` with
the exact command for the (base, head, dataset) you pick, so you
never have to dig through this header.

**4. (Optional) Hardware.** `vit_small` runs CRP in a few GB of VRAM
(CPU works for inspection, slower FV indexing). `vit_dinov3` wants a
≥24 GB GPU; its `attentive`/`block` token-feature cache is ~20 GB.

## Notebook structure

1. Setup (imports, repo paths)
2. Configuration — base × head × train-ds, plus composite
3. Layer name reference — which submodules are hookable
   3b. Single-block computation graph (torchview) + sanity check
4. Load dataset + pick a focal image
5. Build FV index — concept × site combinations
6. Reference samples per concept × site (upstream `plot_grid`)
7. Conditional propagation cascade (inline)
8. Notes & next steps

## 1. Setup

In [ ]:
# %cd ../..
# %ls
from __future__ import annotations
import warnings
from pathlib import Path

# Walk up from the notebook's location to find the repo root —
# only used to point at <repo>/data/. No sys.path manipulation:
# `experiments` and `crp` are installable packages exposed by
# `uv sync` (project.scripts in pyproject.toml).
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent

import torch
import numpy as np
import matplotlib.pyplot as plt

from crp.attribution import CondAttribution
from crp.concepts import HeadConcept, EmbeddingDimConcept, TokenConcept
from zennit_ext import LRPInspectionLayer
from zennit_ext import AttnLRPCombinedComposite
from crp.visualization import FeatureVisualization
from crp.helper import get_layer_names
from crp.image import plot_grid

from experiments.datasets import load as load_dataset
from experiments.models import BASES, HEADS, build_probe
from experiments.viz_unfolded import to_display

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')
print(f'available bases: {list(BASES)}')
print(f'available heads: {list(HEADS)}')

## 2. Configuration — base × head × dataset, plus composite

**Composite** — the recipe validated in `RESEARCH_NOTES.md` Entry 6:

* `matmul_factor_2=True` — bilinear matmul rule (AttnLRP Prop 3.3)
* `alpha=0.5, beta=0.5` — AlphaBeta variant of the bilinear rule
  (Bach 2015 generalised to bilinear — best magnitude control,
  ~19 OOM tighter than the standard `2y+ε` rule on DINOv3 ViT-L)
* `layerscale_uniform=True` — uniform-rule LayerScale γ allocation.
  LayerScale (Touvron et al. CaiT 2021) multiplies each branch by
  γ ≈ 1e-4. Bare backward gives `grad_branch = grad_y · γ` —
  multiplying by γ deflates relevance massively per layer (over
  24 blocks × 2 LayerScales/block this compounds to numerical
  death). `layerscale_uniform=True` wraps `γ * branch` in
  `divide_gradient(., 2)` — the AttnLRP uniform rule (Eq. 7):
  γ (a leaf parameter) absorbs half the relevance, branch's half
  (= R/2) propagates back through the chain. Replaces `×γ`
  dampening with constant `×½` per LayerScale, keeping
  magnitudes alive through deep stacks.
* `residual_lrp='ratio'` — Otsuki ratio split on residual additions
* `use_unfolded_attention=True` — substitute EvaAttention with
  EvaAttentionUnfolded (required for concept conditioning)

**Base × head × dataset** — pick one of each. The model is
(re)built from the same registry the training CLI uses, so swapping
to a different head (e.g. `attentive`) is just changing one variable.

In [ ]:
# === MODEL × TRAIN-DS CHOICE ===
# Pick one (base, head, train_ds). The cell below resolves a checkpoint
# automatically — prefers the most recent OneCycle finetune run under
# `data/runs/finetune_<base>_<train-ds>/<ts>/best.pt`, falls back to
# the legacy probe checkpoint `data/<base>_<head>_probe_<train-ds>.pt`.
# The same backbone receives the same unfolded-attention substitution
# (Eva ↔ EvaAttentionUnfolded, timm ↔ TimmAttentionUnfolded) — concept
# hookable sites (q/k/v_lrp_probe, proj_drop) are uniform across both.

from experiments.train_probe import TRAIN_DATASETS

# Available bases (see `experiments.models.BASES`):
#   'vit_dinov3'  — ViT-L/16 DINOv3 (24 blocks, Eva + LayerScale + RoPE + 4 register tokens)
#   'vit_base'    — ViT-B/16 timm   (12 blocks, no LayerScale)
#   'vit_small'   — ViT-S/16 timm   (12 blocks; our OneCycle-finetuned development substrate)
# Heads: 'linear' | 'attentive' | 'block'.
# Train-ds choices (see TRAIN_DATASETS): 'funny-birds-train-clean',
# 'funny-birds-train-full', 'dsprites', 'colored-mnist-train',
# 'imagenette-train', 'imagenet-val-hf'.
BASE     = 'vit_small'
HEAD     = 'linear'
TRAIN_DS = 'funny-birds-train-clean'

# Resolve the underlying dataset name + kwargs once; used downstream.
DATASET, DATASET_KWARGS = TRAIN_DATASETS[TRAIN_DS]

DATA_ROOT = REPO_ROOT / 'data'

# Checkpoint resolution: latest finetune run wins; probe is the fallback.
import glob as _glob
_finetune_runs = sorted(_glob.glob(str(DATA_ROOT / 'runs' / f'finetune_{BASE}_{TRAIN_DS}' / '*' / 'best.pt')))
_probe_path = DATA_ROOT / f'{BASE}_{HEAD}_probe_{TRAIN_DS}.pt'
if _finetune_runs:
    PROBE_PATH = Path(_finetune_runs[-1])
    print(f'  using OneCycle finetune ckpt: {PROBE_PATH.relative_to(REPO_ROOT)}')
elif _probe_path.is_file():
    PROBE_PATH = _probe_path
    print(f'  using probe ckpt: {PROBE_PATH.relative_to(REPO_ROOT)}')
else:
    # Set explicitly to surface a clean FileNotFoundError in the load cell.
    PROBE_PATH = _probe_path

# Sanity-eval subset size (section 4). Tighter estimate with larger N
# at the cost of more time.
EVAL_SUBSET_SIZE = 200

# Random seed for sample selection.
RANDOM_SEED = 0

# === EXPERIMENT TAG ===
# Optional suffix on the FV cache dir (section 5). Set to a short
# identifier when running a composite variant whose reference
# samples you want kept distinct from the baseline (e.g.
# 'no_ln_canon', 'alpha02beta08'). Empty string → baseline path.
EXPERIMENT_TAG = ''

print(f'base     : {BASE}')
print(f'head     : {HEAD}')
print(f'train_ds : {TRAIN_DS}  ({DATASET} {DATASET_KWARGS})')
print(f'probe    : {PROBE_PATH}')

In [ ]:
# Load the probe checkpoint and rebuild the full model with the same
# `build_probe` registry the training CLI uses. If the probe is
# missing, the cell prints the exact two commands to make it.
if not PROBE_PATH.is_file():
    head_kind = HEADS[HEAD].input_kind
    raise FileNotFoundError(
        f'\nNo checkpoint found for ({BASE}, {HEAD}, {DATASET}).\n\n'
        f'Option A — full OneCycle fine-tune (recommended for vit_small):\n'
        f'  uv run python -m experiments.train_probe finetune \\\n'
        f'    --from-scratch --base {BASE} --dataset {DATASET} \\\n'
        f'    --head {HEAD} --epochs 25 --scheduler onecycle \\\n'
        f'    --backbone-lr 5e-4 --head-lr 5e-3 --llrd 0.7 \\\n'
        f'    --randaugment --label-smoothing 0.1\n\n'
        f'Option B — frozen-backbone probe (DINOv3 style):\n'
        f'  uv run train-probe cache {BASE} {DATASET} --kind {head_kind}\n'
        f'  uv run train-probe train {BASE} {HEAD} {DATASET}\n'
    )
ckpt = torch.load(PROBE_PATH, map_location=DEVICE, weights_only=False)
print(f'probe trained on : {ckpt["dataset"]}'
      f' ({ckpt["num_classes"]} classes)')
_va, _va5 = ckpt.get('val_acc'), ckpt.get('val_acc5')
print(f'val_acc          : {_va:.4f}' if _va is not None else 'val_acc          : (unreported — see sanity-eval below)')
print(f'val_acc5         : {_va5:.4f}' if _va5 is not None else 'val_acc5         : (unreported)')

In [ ]:
from timm.data import resolve_data_config, create_transform
import torch.nn.functional as F
from torchvision.transforms import functional as TF

# Build the full Probe (frozen base + trainable head) via the same
# registry used by the CLI. Backbone is loaded fresh from timm via
# build_probe → Base.__init__; trained head weights come from the ckpt.
model = build_probe(
    base=ckpt['base'], head=ckpt['head'],
    num_classes=ckpt['num_classes'],
    head_kwargs=ckpt.get('head_kwargs', {}),
).eval().to(DEVICE)
model.head.load_state_dict(ckpt['head_state_dict'])
if 'backbone_state_dict' in ckpt:
    # Fine-tuned or externally-trained backbone (e.g. setup_funnybirds_vit_base.py
    # or train_probe finetune_cmd output) overrides whatever the Base class
    # loaded at construction.
    model.backbone.load_state_dict(ckpt['backbone_state_dict'])
    print('  loaded backbone weights from ckpt (pretrained / finetuned)')
for p in model.parameters():
    p.requires_grad_(False)

# Build the eval transform AND the per-batch normalize callable. The
# dataset transform produces unnormalized [0, 1] tensors (display-
# ready, uniform across DataLoader / FeatureVisualization / Lightning
# / raw forward); normalize is applied at the forward boundary so the
# model sees its expected input distribution. This split lets external
# pretrained checkpoints (e.g. visinf vit_base, trained without
# normalize) coexist with timm-default models in the same notebook
# without any denormalize indirection for display.
TRANSFORM_SPEC = ckpt.get('transform_spec', 'timm_default')
if 'transform_spec' not in ckpt:
    print('⚠ ckpt has no `transform_spec` field — falling back to timm_default.')
    if ckpt.get('base') == 'vit_base' and ckpt.get('dataset') == 'funny_birds':
        print('  Your payload looks like the visinf vit_base FunnyBirds checkpoint.')
        print('  Re-run the setup script to refresh the payload with transform_spec:')
        print('    uv run python experiments/scripts/setup_funnybirds_vit_base.py --force')
        print('  Then re-run this cell.')

if TRANSFORM_SPEC == 'timm_default':
    # Resize + crop + ToTensor from timm cfg; mean/std overridden to
    # identity so the timm transform's normalize step is a no-op.
    # The actual normalize stats (read from the registered cfg) ship
    # to the model in a separate callable.
    cfg = resolve_data_config({}, model=model.backbone)
    _resize_cfg = {**cfg, 'mean': (0.0, 0.0, 0.0), 'std': (1.0, 1.0, 1.0)}
    transform = create_transform(**_resize_cfg, is_training=False)
    _mean = torch.tensor(cfg['mean']).view(1, -1, 1, 1).to(DEVICE)
    _std  = torch.tensor(cfg['std']).view(1, -1, 1, 1).to(DEVICE)
    def normalize(x):
        return (x - _mean) / _std
    print(f'transform  : timm_default (resize-only; normalize mean={cfg["mean"]}, std={cfg["std"]})')

elif TRANSFORM_SPEC == 'visinf_funnybirds_vit_base':
    # visinf/funnybirds-framework trains its ViT with `transforms=None`
    # in train.py (only `torchvision.transforms.ToTensor` → [0,1])
    # — NO ImageNet/JFT normalize. Their model wrapper resizes 256→224
    # via `F.interpolate(x, (224, 224))` (defaults to nearest). We use
    # bilinear here for cleaner image resampling — both give 0.98–0.99
    # top-1 on the FunnyBirds test set (within sampling noise of the
    # checkpoint's own `best_acc1=98.0`); the (0.5,0.5,0.5)-normalize +
    # crop_pct=0.9 timm default drops it to ~0.85. The model expects
    # raw [0, 1] inputs → `normalize` is the identity.
    def transform(pil):
        t = TF.to_tensor(pil)[:3]  # drops alpha, keeps [0,1] floats
        t = F.interpolate(t.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False).squeeze(0)
        return t
    def normalize(x):
        return x
    print('transform  : visinf_funnybirds_vit_base (resize-only; no-normalize)')

else:
    raise ValueError(f'unknown transform_spec {TRANSFORM_SPEC!r} — register a branch in the build-model cell')

_npt = int(getattr(model, 'num_prefix_tokens', 1))
print(f'base       : {ckpt["base"]}')
print(f'head       : {ckpt["head"]} kwargs={ckpt.get("head_kwargs", {})}')
print(f'embed_dim  : {model.embed_dim}')
print(f'num_blocks : {len(model.blocks)}')
print(f'num_heads  : {model.blocks[0].attn.num_heads}')
print(f'head_dim   : {model.blocks[0].attn.head_dim}')
if _npt > 1:
    print(f'num_prefix : {_npt} (1 cls + {_npt - 1} register)')
else:
    print(f'num_prefix : {_npt} (cls only)')

# Read structural dims from the model — concept classes need num_heads
# (HeadConcept / EmbeddingDimConcept slice embed_dim per head).
NUM_HEADS = int(model.blocks[0].attn.num_heads)
HEAD_DIM  = int(model.blocks[0].attn.head_dim)
EMBED_DIM = int(model.embed_dim)
NUM_PREFIX_TOKENS = _npt

# Per-section layer-name discovery uses `get_layer_names` (upstream).
# All hookable LRP-inspection sites are instances of
# `LRPInspectionLayer`; `proj_drop` is a stock `nn.Dropout` and is
# enumerated separately in the sections that need it. See section 3 for
# the discovery printout.

In [ ]:
# === COMPOSITE — built explicitly so every rule is editable ===
# A `layer_map` maps module classes → LRP rules (zennit Hooks); `canonizers`
# do the structural forward rewrites. zennit's LayerMapComposite wires them.
# This is exactly what `AttnLRPCombinedComposite` builds internally, spelled
# out so you can swap a rule or drop a canonizer and re-run.
import torch.nn as nn
from zennit.rules import Epsilon, Pass
from zennit.composites import LayerMapComposite
from zennit_ext import (
    # unfolded-attention module types (rule targets)
    BilinearMatmul, SoftmaxAlongLastDim, ScaleByConstant,
    ResidualAdd, UniformAdd, LayerScaleMul,
    # LRP rules as zennit Hooks
    AlphaBetaMatmul, ResidualRatio, Uniform,
    # structural forward-rewrite canonizers
    LayerNormForwardCanonizer, DropoutPassthroughCanonizer,
    TimmBlockResidualCanonizer, EvaBlockResidualCanonizer,
    EvaAttentionSubstitutionCanonizer, TimmAttentionSubstitutionCanonizer,
)

EPSILON     = 1e-6
ALPHA, BETA = 0.5, 0.5

# ─── LAYER MAP — per-module-class LRP rule (zennit Hooks) ───────────
layer_map = [
    # attention internals (exposed by the substitution canonizers below)
    (BilinearMatmul,      AlphaBetaMatmul(alpha=ALPHA, beta=BETA, epsilon=EPSILON)),
    (SoftmaxAlongLastDim, Pass()),   # identity rule (AttnLRP Eq. 9)
    (ScaleByConstant,     Pass()),   # graph constant absorbs no relevance
    # residual / layerscale / pos-embed adds (modules emitted by canonizers)
    (ResidualAdd,         ResidualRatio(epsilon=EPSILON)),  # Otsuki ratio split
    (UniformAdd,          Uniform(factor=2)),               # symmetric residual + x+pos_embed
    (LayerScaleMul,       Uniform(factor=2)),               # gamma absorbs half
    # generic layers
    (nn.Linear,           Epsilon(epsilon=EPSILON)),
    (nn.Conv2d,           Epsilon(epsilon=EPSILON)),
    (nn.GELU,             Pass()),
    (nn.LayerNorm,        Pass()),   # std stop-grad done by the canonizer below
    (nn.Dropout,          Pass()),
    (nn.Identity,         Pass()),
]

# ─── CANONIZERS — structural forward rewrites only ──────────────────
canonizers = [
    LayerNormForwardCanonizer(),
    DropoutPassthroughCanonizer(),
    TimmBlockResidualCanonizer(residual_rule='ratio'),
    EvaBlockResidualCanonizer(residual_rule='ratio', layerscale_uniform=True),
    EvaAttentionSubstitutionCanonizer(block_indices=None),
    TimmAttentionSubstitutionCanonizer(block_indices=None),
]

composite = LayerMapComposite(layer_map=layer_map, canonizers=canonizers)
attribution = CondAttribution(model)
print(f'composite: {len(canonizers)} canonizers, {len(layer_map)} layer_map entries')

### Alternative composite — LXT paper parametrisation (reproducibility)

Closest match in this codebase to the LRP recipe proposed by LXT
(*LRP eXplains Transformers*, Achtibat et al. — github.com/
rachtibat/LRP-eXplains-Transformers). Use this to sanity-check
the zennit implementation against LXT-style heatmaps:

| Module class           | Rule                                  | Params           |
|---|---|---|
| `nn.Conv2d`            | zennit Gamma                          | γ=0.25, stab=1e-6 |
| `nn.Linear`            | zennit Gamma                          | γ=0.10, stab=1e-6 |
| `nn.GELU`              | LXT identity (R_in = R_out)            | —                |
| `nn.LayerNorm`         | LXT identity, stop-grad on std         | model eps=1e-6   |
| `BilinearMatmul` (qk)  | CP-LRP: stop-grad Q,K → no propagation | —                |
| `BilinearMatmul` (ctx) | autograd default (`R_v = weightsᵀ R_y`) | —                |

Key difference vs the AlphaBeta(0.5, 0.5) composite above:
**CP-LRP routes relevance through the value path only**.
`QInspectionLayer` and `KInspectionLayer` are mapped to
`StopGradient()` — a zennit `Hook` that zeros backward relevance
at the Q and K probes. With no gradient flowing through Q or K,
the softmax map downstream becomes a graph constant, so
autograd's natural backward on `context = weights @ v` delivers
exactly the LXT CP-LRP value attribution (`R_v = weightsᵀ @ R_out`)
— no custom backward kernel on `BilinearMatmul` needed. The V
probe stays as a pure identity (`VInspectionLayer`) so the
value-path relevance is untouched.

Implementation notes (where we extrapolate beyond the table):

* **Residual additions** are not in the LXT table. We keep the
  ratio rule (Otsuki split) — same as the main composite. Pure
  autograd would double-count relevance at every block-level `+`;
  LXT's reference code applies a conservative residual rule too.
  Swap `residual_rule='ratio'` → `'symmetric'` for the factor-2
  uniform variant if matching a different LXT setup.
* **LayerScale (DINOv3 / Eva blocks).** Not in the LXT table
  either; vit_base / vit_small don't have it, so for those
  backbones the `layerscale_uniform` flag is a no-op. Left at
  `True` for parity on DINOv3.
* **Softmax / scale-by-constant** map to `Pass` in the layer_map —
  harmless extras under CP-LRP (both ops are downstream of the
  detached Q,K and carry no gradient anyway).

**To activate**: rebind `composite = composite_lxt` and re-run
from section 3 onward. Use `EXPERIMENT_TAG = 'lxt'` (section 2)
so FV reference samples land in a separate cache.

In [ ]:
# === COMPOSITE — LXT paper recipe (CP-LRP attention + gamma-rule linears) ===
from zennit.rules import Gamma
from zennit_ext import (
    SoftmaxAlongLastDim, ScaleByConstant, ResidualAdd, UniformAdd, LayerScaleMul,
    ResidualRatio, Uniform, QInspectionLayer, KInspectionLayer, StopGradient,
    LayerNormForwardCanonizer, DropoutPassthroughCanonizer,
    TimmBlockResidualCanonizer, EvaBlockResidualCanonizer,
    EvaAttentionSubstitutionCanonizer, TimmAttentionSubstitutionCanonizer,
)

LXT_LINEAR_GAMMA = 0.10
LXT_CONV_GAMMA   = 0.25
LXT_EPSILON      = 1e-6

# CP-LRP attention: Q,K probes -> StopGradient (kill those paths). With no
# Q/K gradient the softmax map is a graph constant, so autograd's default
# backward on `weights @ v` delivers R_v = weights^T R_out. NO AlphaBeta on
# BilinearMatmul here (that is the AlphaBeta composite above).
lxt_layer_map = [
    (nn.Linear,           Gamma(gamma=LXT_LINEAR_GAMMA)),
    (nn.Conv2d,           Gamma(gamma=LXT_CONV_GAMMA)),
    (nn.GELU,             Pass()),
    (nn.LayerNorm,        Pass()),
    (nn.Dropout,          Pass()),
    (SoftmaxAlongLastDim, Pass()),
    (ScaleByConstant,     Pass()),
    (ResidualAdd,         ResidualRatio(epsilon=LXT_EPSILON)),
    (UniformAdd,          Uniform(factor=2)),
    (LayerScaleMul,       Uniform(factor=2)),
    (QInspectionLayer,    StopGradient()),  # CP-LRP: kill Q-path relevance
    (KInspectionLayer,    StopGradient()),  # CP-LRP: kill K-path relevance
    (nn.Identity,         Pass()),          # incl. VInspectionLayer (untouched)
]

lxt_canonizers = [
    LayerNormForwardCanonizer(),
    DropoutPassthroughCanonizer(),
    TimmBlockResidualCanonizer(residual_rule='ratio'),
    EvaBlockResidualCanonizer(residual_rule='ratio', layerscale_uniform=True),
    EvaAttentionSubstitutionCanonizer(block_indices=None),
    TimmAttentionSubstitutionCanonizer(block_indices=None),
]

composite_lxt = LayerMapComposite(layer_map=lxt_layer_map, canonizers=lxt_canonizers)
print(f'composite_lxt: {len(lxt_canonizers)} canonizers, {len(lxt_layer_map)} layer_map entries')

# To evaluate LXT side-by-side: composite = composite_lxt ; EXPERIMENT_TAG = "lxt"

## 3. Layer name reference (which submodules are hookable)

The unfolded attention exposes three `LRPInspectionLayer` sites per
block — `q_lrp_probe`, `k_lrp_probe`, `v_lrp_probe` — placed right
after the qkv-split, before the per-head reshape. All three carry
the SAME tensor shape `(B, N, embed_dim)` as `proj_drop` (the
attention output). The three concept classes operate on that one
shape contract:

| Concept | What it selects | Hookable at |
|---|---|---|
| `HeadConcept(num_heads)`        | one head's slice of `embed_dim` (sums over `head_dim` adjacent indices) | any 3D site: `q_lrp_probe`, `k_lrp_probe`, `v_lrp_probe`, `proj_drop` |
| `EmbeddingDimConcept(num_heads)`| one single dim of `embed_dim` (finer than per-head) | same as above |
| `TokenConcept()`                | one token position | `proj_drop` (token-position attribution most natural at the block output) |

All three accept an optional `token_filter=slice(...)` to restrict
the token axis (e.g. `slice(num_prefix_tokens, None)` for spatial-
only). Default = include all tokens (cls + register + spatial).

Discover layer names via `crp.helper.get_layer_names(model, [LRPInspectionLayer])`
for the probe sites; raw `model.named_modules()` for `proj_drop` /
`context` etc.

In [ ]:
# Enumerate all LRPInspectionLayer sites in the model. Three per
# block: q_lrp_probe / k_lrp_probe / v_lrp_probe.
with composite.context(model):
    probe_layer_names = get_layer_names(model, [LRPInspectionLayer])
print(f'LRPInspectionLayer sites: {len(probe_layer_names)} total')
for n in probe_layer_names[:6]:
    print(f'  {n}')
if len(probe_layer_names) > 6:
    print(f'  ... ({len(probe_layer_names) - 6} more)')

print()
print(f'NUM_HEADS={NUM_HEADS}, HEAD_DIM={HEAD_DIM}, EMBED_DIM={EMBED_DIM}, NUM_PREFIX_TOKENS={NUM_PREFIX_TOKENS}')

### Single-block computation graph

Visualize one attention block's unfolded internals as a directed
graph. The composite context substitutes the stock attention with
`EvaAttentionUnfolded` / `TimmAttentionUnfolded`, exposing every
atomic op (`qkv`, `q_lrp_probe` / `k_lrp_probe` / `v_lrp_probe`,
`q_norm`, `scale_q`, `qk_scores`, `softmax`, `context`, `proj`, …)
as a named child module. The named
submodules in the rendered graph are exactly the strings you'd pass
to `record_layer=[...]` or use in a condition dict.

Renders via [`torchview`](https://github.com/mert-kurttutan/torchview),
which calls system Graphviz to produce an SVG. If `graphviz` isn't
installed, falls back to printing the named-submodule tree.
Install graphviz with `apt install graphviz` / `brew install graphviz`.

In [ ]:
BLOCK_TO_VIZ = 0  # block index to render — same graph for all blocks

try:
    from torchview import draw_graph
    _can_torchview = True
except ImportError:
    print('torchview not installed; run `uv pip install torchview` and re-run.')
    _can_torchview = False

if _can_torchview:
    # Substitute the attention with its unfolded variant via the composite
    # context — that's the graph we actually run attribution against.
    with composite.context(model) as modified:
        block = modified.backbone.blocks[BLOCK_TO_VIZ]
        # Probe the right token-sequence shape from the model.
        _N = _npt + int(getattr(model, 'num_patches', 196))
        _x = torch.randn(1, _N, model.embed_dim, device=DEVICE)
        try:
            g = draw_graph(
                block, input_data=_x,
                depth=3, expand_nested=True,
                graph_name=f'block_{BLOCK_TO_VIZ}',
                hide_module_functions=False,
            )
            # Render to in-memory SVG. Requires system `graphviz` (the
            # `dot` binary). On a missing binary the call raises;
            # we catch and fall back to a structural print below.
            from IPython.display import display, SVG
            svg_str = g.visual_graph.pipe(format='svg').decode('utf-8')
            display(SVG(svg_str))
        except Exception as e:
            print(f'graphviz render failed ({type(e).__name__}: {e!s}).')
            print(f'Falling back to named-submodule tree of block {BLOCK_TO_VIZ}:')
            print('  ' + '\n  '.join(
                f'{name}: {type(m).__name__}'
                for name, m in block.named_modules() if name
            ))

#### Canonized-block sanity check

The torchview render can look like it has dead ends — that's a
**graph-layout artifact**, not a real disconnection. This cell is
the trustworthy ground truth:

1. **Forward parity** — the substituted block (unfolded attention)
   must produce the same output tensor as the stock block on the
   same input (within fp32 noise).
2. **No dead ends** — every named submodule's output must have a
   nonzero gradient w.r.t. the block output. If anything has zero
   or `None` gradient, it's dead and a real bug.

In [ ]:
import torch as _torch_chk

with _torch_chk.no_grad():
    _stock_block = model.backbone.blocks[BLOCK_TO_VIZ]
    _x_stock = _torch_chk.randn(1, _N, model.embed_dim, device=DEVICE)
    _y_stock = _stock_block(_x_stock)

with composite.context(model) as _modified_for_check:
    _sub_block = _modified_for_check.backbone.blocks[BLOCK_TO_VIZ]

    # 1. Forward parity
    with _torch_chk.no_grad():
        _y_sub = _sub_block(_x_stock)
    _max_diff = (_y_sub - _y_stock).abs().max().item()
    _stock_sum, _sub_sum = _y_stock.sum().item(), _y_sub.sum().item()
    _ok_fwd = _max_diff < 1e-2 * max(abs(_stock_sum), 1.0)
    print(f'Forward parity: stock sum={_stock_sum:.4f}, substituted sum={_sub_sum:.4f}')
    print(f'  max element-wise diff = {_max_diff:.3e}  → {"OK" if _ok_fwd else "⚠ MISMATCH"}')

    # 2. Dead-end check — register forward hooks on every named submodule
    _captured = {}
    _handles = []
    for _name, _m in _sub_block.named_modules():
        if not _name:
            continue
        def _hook(_mod, _inp, _out, __n=_name):
            if isinstance(_out, _torch_chk.Tensor) and _out.requires_grad:
                _out.retain_grad()
                _captured[__n] = _out
        _handles.append(_m.register_forward_hook(_hook))
    try:
        _x_grad = _torch_chk.randn(1, _N, model.embed_dim, device=DEVICE, requires_grad=True)
        _y_grad = _sub_block(_x_grad)
        _y_grad.sum().backward()
    finally:
        for _h in _handles:
            _h.remove()
    _dead = [
        n for n, t in _captured.items()
        if t.grad is None or t.grad.abs().sum().item() == 0
    ]
    print(f'\nDead-end check on {len(_captured)} hookable submodules:')
    if _dead:
        print(f'  ⚠ {len(_dead)} dead end(s) — these submodules do NOT contribute to the block output:')
        for n in _dead:
            print(f'    {n}')
    else:
        print(f'  ✓ all {len(_captured)} submodules contribute to the block output')

## 4. Load dataset + pick a focal image

The chosen `DATASET` is loaded via the unified `load(name, ...)`
dispatcher in `experiments/datasets/`. Each dataset module handles
its own download/extract/setup automatically. The focal image is
the first correctly-classified sample under the trained probe;
everything below attributes against this image's predicted class.

In [ ]:
# Build the dataset variants used downstream. For FunnyBirds we expose
# three named handles so it's easy to switch what the FV cache (sec 9)
# and the cascade (sec 11) operate on:
#
#   dataset_test          : held-out test split (500 imgs, 0% ablations)
#   dataset_train_clean   : train filtered to intact birds (~29k imgs)
#   dataset_train_ablated : full train, includes part-ablations (~50k,
#                           ~41% have one or more body parts replaced
#                           with 'placeholder' renders).
#
# Downstream cells read `dataset` — change the assignment below to
# switch the FV indexing pool. The sanity-eval cell uses
# `dataset_eval` independently (always the clean test split for
# FunnyBirds — matches the advertised accuracy figure).
if DATASET == 'funny_birds':
    dataset_test          = load_dataset('funny_birds', transform=transform, split='test')
    dataset_train_clean   = load_dataset('funny_birds', transform=transform, split='train', clean_only=True)
    dataset_train_ablated = load_dataset('funny_birds', transform=transform, split='train', clean_only=False)
    dataset_eval = dataset_test
    # Default: ablated train — gives the FV cache access to part-
    # ablation samples, so the resulting concept references reveal
    # whether any head/channel fires specifically on missing-part
    # renders. Switch to `dataset_train_clean` for a baseline FV that
    # mirrors the test distribution.
    dataset = dataset_train_ablated
    print(f'  dataset_test          : {len(dataset_test):>6d} imgs')
    print(f'  dataset_train_clean   : {len(dataset_train_clean):>6d} imgs')
    print(f'  dataset_train_ablated : {len(dataset_train_ablated):>6d} imgs')
else:
    _load_kwargs = {
        'dsprites':        dict(target='shape'),
        'imagenette':      dict(split='val'),
        'imagenet_val_hf': dict(),
    }[DATASET]
    dataset_eval = dataset = load_dataset(DATASET, transform=transform, **_load_kwargs)
print(f'downstream `dataset`      : {len(dataset):>6d} imgs')
print(f'sanity-eval `dataset_eval`: {len(dataset_eval):>6d} imgs')

### Sanity-eval: does the loaded model classify this dataset?

Run the model on a small slice of `dataset_eval` (the held-out test
split for FunnyBirds, which by construction has 0% ablations). A
near-chance number here means the checkpoint didn't load correctly
(wrong head shape, missing backbone weights, base/dataset mismatch, or
a transform mismatch between the checkpoint's training pipeline and
our eval pipeline). Bump `EVAL_SUBSET_SIZE` (in section 2) for tighter
estimates.

In [ ]:
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

n_eval = min(EVAL_SUBSET_SIZE, len(dataset_eval))
_loader = DataLoader(
    Subset(dataset_eval, list(range(n_eval))),
    batch_size=32, shuffle=False, num_workers=2,
)
_correct = _total = 0
with torch.no_grad():
    for _x, _y in tqdm(_loader, desc='sanity-eval', unit='batch'):
        # Dataset yields unnormalized tensors; normalize at the forward
        # boundary.
        _logits = model(normalize(_x.to(DEVICE)))
        _correct += (_logits.argmax(-1).cpu() == _y).sum().item()
        _total   += _y.numel()
print(f'Sanity-eval top-1 over {_total} samples of dataset_eval: {_correct/_total:.4f}')
if 'val_acc' in ckpt and ckpt['val_acc'] is not None:
    print(f'  (advertised val_acc in payload: {ckpt["val_acc"]:.4f})')

### Pick the focal image

In [ ]:
# `focal_image` holds the UNNORMALIZED [0, 1] tensor — display-ready,
# safe to pass to viz_unfolded funcs (which `preprocess_fn=normalize`).
# Model forwards inside this cell normalize at the boundary.
rng = np.random.default_rng(RANDOM_SEED)
stride = max(1, len(dataset) // 30)
focal_image = focal_class = focal_index = None
for i in range(0, len(dataset), stride):
    x_, y_ = dataset[i]
    x_dev = x_.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(normalize(x_dev)).argmax(-1).item()
    if pred == int(y_):
        focal_image = x_dev.detach().requires_grad_(True)
        focal_class = pred
        focal_index = i
        break
if focal_image is None:
    raise RuntimeError(
        f'No correctly-classified sample found in first {len(dataset)//stride} '
        f'strided samples — probe accuracy may be too low. Re-train it with '
        f'more epochs or check dataset compatibility.'
    )

print(f'focal image: dataset index {focal_index}, class {focal_class}')
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(to_display(focal_image)); ax.axis('off')
ax.set_title(f'class {focal_class}')
plt.show()

### Plain attribution (no concept conditioning) for reference

In [ ]:
# Attribution runs on the NORMALIZED tensor — that's what the model
# expects internally. The display tensor stays unnormalized.
x_run = normalize(focal_image).detach().clone().requires_grad_(True)
res = attribution(x_run, [{'y': [focal_class]}], composite)
hm = res.heatmap[0].detach().cpu().numpy()
if hm.ndim == 3 and hm.shape[0] == 3:
    hm = hm.sum(axis=0)
fig, (ax_img, ax_hm) = plt.subplots(1, 2, figsize=(6, 3))
ax_img.imshow(to_display(focal_image)); ax_img.axis('off')
ax_img.set_title(f'class {focal_class}', fontsize=9)
vmax = abs(hm).max() or 1.0
ax_hm.imshow(hm, cmap='seismic', vmin=-vmax, vmax=vmax); ax_hm.axis('off')
ax_hm.set_title('plain attribution', fontsize=9)
plt.tight_layout(); plt.show()

## 5. Build FV index for reference-sample retrieval

Three concepts (`HeadConcept`, `EmbeddingDimConcept`, `TokenConcept`)
× target sites (the LRPInspectionLayer probes + `proj_drop`). FV
indexes the relevance of each concept id over the dataset and caches
the top-N images that most activate it.

**Cache layout** (unique per `(base, head, dataset)` so multiple
combinations coexist; also safe across parallel notebook kernels):

```
data/fv_cache/<base>_<head>_<train-ds>/<tag>/
    RelMax_*/...      # top-N indices, max-relevance
    ActMax_*/...      # top-N indices, max-activation
```

**Re-running the cell is safe**: if `<tag>/RelMax_*/*_data.npy`
already exists, the indexing step is skipped and the FV object is
rebound to the on-disk index. To force a rebuild, delete the matching
tag folder (or the whole `<base>_<head>_<dataset>` folder).

In [ ]:
# Cache root keyed by (base, head, dataset) — switching BASE/HEAD/DATASET
# in section 2 routes to a different cache, so combinations don't
# collide. Independent caches are also safe across parallel notebook
# kernels (each writes only into its own path).
# An optional EXPERIMENT_TAG (set in section 2) becomes a suffix on
# the cache dir so different composite variants can index in
# parallel without overwriting each other.
_suffix = f'_{EXPERIMENT_TAG}' if EXPERIMENT_TAG else ''
FV_CACHE_DIR = REPO_ROOT / 'data' / 'fv_cache' / f'{BASE}_{HEAD}_{TRAIN_DS}{_suffix}'
FV_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'cache dir: {FV_CACHE_DIR.relative_to(REPO_ROOT)}')

# Discover all 3D inspection sites: the LRPInspectionLayer probes
# (q/k/v_lrp_probe per block) plus proj_drop (per block).
with composite.context(model):
    probe_layer_names = get_layer_names(model, [LRPInspectionLayer])
proj_drop_layer_names = [
    f'backbone.blocks.{i}.attn.proj_drop' for i in range(len(model.blocks))
]

# Quick defaults: index a single concept × single layer for each
# concept type. Comment / uncomment / extend per your analysis. Adding
# more layers per fv_specs entry just grows the layer_map of one
# FeatureVisualization — same one forward+backward pass per image.
_LAST_BLOCK = len(model.blocks) - 1
_LAST_PROJ  = f'backbone.blocks.{_LAST_BLOCK}.attn.proj_drop'
_LAST_QPROBE = f'backbone.blocks.{_LAST_BLOCK}.attn.q_lrp_probe'
_LAST_KPROBE = f'backbone.blocks.{_LAST_BLOCK}.attn.k_lrp_probe'
_LAST_VPROBE = f'backbone.blocks.{_LAST_BLOCK}.attn.v_lrp_probe'

fv_specs = [
    # (cache-tag, concept-instance, list-of-target-layers)
    ('head_at_proj',     HeadConcept(num_heads=NUM_HEADS),         [_LAST_PROJ]),
    ('head_at_qprobe',   HeadConcept(num_heads=NUM_HEADS),         [_LAST_QPROBE]),
    ('head_at_kprobe',   HeadConcept(num_heads=NUM_HEADS),         [_LAST_KPROBE]),
    ('head_at_vprobe',   HeadConcept(num_heads=NUM_HEADS),         [_LAST_VPROBE]),
    ('embdim_at_proj',   EmbeddingDimConcept(num_heads=NUM_HEADS), [_LAST_PROJ]),
    ('token_at_proj',    TokenConcept(),                            [_LAST_PROJ]),
    # ── Cascade-style: HeadConcept across multiple depths ────────────
    # ('head_cascade', HeadConcept(num_heads=NUM_HEADS),
    #     [f'backbone.blocks.{i}.attn.proj_drop' for i in [_LAST_BLOCK, _LAST_BLOCK*3//4, _LAST_BLOCK//2, _LAST_BLOCK//4, 0]]),
]

from tqdm.auto import tqdm

# Outer bar tracks fv_specs entries; the inner fv.run() shows its own
# per-image progress bar from upstream crp/visualization.py.
fv_results = {}
_outer = tqdm(fv_specs, desc='FV specs', unit='spec')
for tag, concept, layers in _outer:
    _outer.set_postfix_str(tag)
    cache_path = FV_CACHE_DIR / tag
    fv = FeatureVisualization(
        attribution=attribution, dataset=dataset,
        layer_map={layer: concept for layer in layers},
        preprocess_fn=normalize,
        path=str(cache_path), device=DEVICE,
    )
    has_index = cache_path.is_dir() and all(
        list(cache_path.rglob(f'{layer}_data.npy')) for layer in layers
    )
    if has_index:
        tqdm.write(f'skip {tag}: index present at {cache_path}')
    else:
        tqdm.write(f'index {tag} → {len(layers)} layer(s) over {len(dataset)} imgs ...')
        fv.run(composite, 0, len(dataset), batch_size=4, checkpoint=999)
    fv_results[tag] = (fv, concept, layers)
print('FV indexing complete')

## 6. Reference samples per concept × site

Upstream-style flow: pick top-K concept ids from the FV's indexed
scores, fetch their reference samples, plot with `plot_grid`. One
cell per concept × site you indexed. Repeat the pattern for any
additional `(tag, concept_ids)` you care about.

In [ ]:
# HeadConcept @ proj_drop — what does each head's contribution to the
# attention output look like?
fv, concept, layers = fv_results['head_at_proj']
layer = layers[0]
head_ids = list(range(NUM_HEADS))   # show all heads
ref = fv.get_max_reference(head_ids, layer, 'relevance', (0, 8), composite=composite)
print(f'HeadConcept at {layer} — top-8 reference samples per head ({NUM_HEADS} heads)')
_ = plot_grid(ref, figsize=(12, 3 * NUM_HEADS), padding=False)

In [ ]:
# Same HeadConcept, hooked at q_lrp_probe — ‘which input pixels',
# 'populated this head’s query subspace?'
fv, concept, layers = fv_results['head_at_qprobe']
layer = layers[0]
ref = fv.get_max_reference(list(range(NUM_HEADS)), layer, 'relevance', (0, 8), composite=composite)
print(f'HeadConcept at {layer}')
_ = plot_grid(ref, figsize=(12, 3 * NUM_HEADS), padding=False)

In [ ]:
# Same idea for v_lrp_probe — value subspace.
fv, concept, layers = fv_results['head_at_vprobe']
layer = layers[0]
ref = fv.get_max_reference(list(range(NUM_HEADS)), layer, 'relevance', (0, 8), composite=composite)
print(f'HeadConcept at {layer}')
_ = plot_grid(ref, figsize=(12, 3 * NUM_HEADS), padding=False)

In [ ]:
# EmbeddingDimConcept at proj_drop — top-K dims.
fv, concept, layers = fv_results['embdim_at_proj']
layer = layers[0]
TOP_K = 8
# Pick the top-K dims by abs-sum of relevance from FV's pre-computed
# top-N table. (FV already sorts ids by relevance internally; here we
# just sample the first TOP_K dim ids 0..TOP_K-1 for a deterministic demo.)
dim_ids = list(range(TOP_K))
ref = fv.get_max_reference(dim_ids, layer, 'relevance', (0, 8), composite=composite)
print(f'EmbeddingDimConcept at {layer} — dims {dim_ids}')
_ = plot_grid(ref, figsize=(12, 3 * TOP_K), padding=False)

In [ ]:
# TokenConcept at proj_drop — per-token-position attribution.
# Default token_filter = slice(None) → ids index into the FULL token
# axis. For DINOv3 (5 prefix tokens): id 0 = cls, ids 1..4 = register,
# ids 5+ = patches. For vit_base (1 prefix): id 0 = cls, ids 1+ = patches.
fv, concept, layers = fv_results['token_at_proj']
layer = layers[0]
# Show the prefix tokens (cls + any register tokens).
token_ids = list(range(NUM_PREFIX_TOKENS))
ref = fv.get_max_reference(token_ids, layer, 'relevance', (0, 8), composite=composite)
labels = ['cls'] + [f'reg{i}' for i in range(NUM_PREFIX_TOKENS - 1)]
print(f'TokenConcept at {layer} — prefix-token positions: {labels}')
_ = plot_grid(ref, figsize=(12, 3 * NUM_PREFIX_TOKENS), padding=False)

## 7. Conditional propagation cascade (inline)

Walk attention layers from deep to shallow. At each layer, condition
on the top-K most relevant heads, accumulate the conditioning, render
one heatmap row per layer. No helper function — the loop is right
here so you can tinker.

In [ ]:
import matplotlib.pyplot as plt

# Cascade depths: deep → shallow. Use proj_drop sites so we get the
# attention block's full output.
_L = _LAST_BLOCK
cascade_layer_names = [
    f'backbone.blocks.{i}.attn.proj_drop'
    for i in [_L, _L * 3 // 4, _L // 2, _L // 4, 0]
]

cascade_concept = HeadConcept(num_heads=NUM_HEADS)
TOP_K = 4

# Focal image (already picked above) → re-run forward + per-layer
# conditional attribution. Built the normalize fn in the build-model
# cell.
x_focal = normalize(focal_image).detach().clone().requires_grad_(True)

fig, axes = plt.subplots(
    len(cascade_layer_names), TOP_K + 1,
    figsize=(2.5 * (TOP_K + 1), 2.5 * len(cascade_layer_names)),
    squeeze=False,
)
extra_conditions = {}   # accumulates as we walk deep → shallow
selected = {}
for row, layer in enumerate(cascade_layer_names):
    # Per-head relevance scores under the cumulative conditioning.
    cond = {'y': [focal_class], **extra_conditions}
    res = attribution(
        x_focal, [cond], composite,
        mask_map=cascade_concept.mask, record_layer=[layer],
        exclude_parallel=False,
    )
    rel = res.relevances[layer]   # (1, N, embed_dim)
    head_scores = cascade_concept.attribute(rel, abs_norm=False)[0]   # (NUM_HEADS,)
    top_heads = torch.argsort(head_scores.abs(), descending=True)[:TOP_K].tolist()
    selected[layer] = top_heads

    # Show focal image on the left, then one heatmap per top head.
    axes[row, 0].imshow(to_display(focal_image)); axes[row, 0].axis('off')
    axes[row, 0].set_title(layer.replace('backbone.', ''), fontsize=7, loc='left', family='monospace')
    for col, h in enumerate(top_heads, start=1):
        x_h = normalize(focal_image).detach().clone().requires_grad_(True)
        res_h = attribution(
            x_h, [{layer: [h], 'y': [focal_class], **extra_conditions}],
            composite, mask_map=cascade_concept.mask,
            exclude_parallel=False,
        )
        hm = res_h.heatmap[0].detach().cpu().numpy()
        if hm.ndim == 3:
            hm = hm.sum(axis=0)
        vmax = abs(hm).max() or 1.0
        axes[row, col].imshow(hm, cmap='seismic', vmin=-vmax, vmax=vmax)
        axes[row, col].axis('off')
        axes[row, col].set_title(f'H{h}', fontsize=7)

    # Add this layer's selection to extra_conditions for the next (shallower) layer.
    extra_conditions[layer] = top_heads

fig.suptitle(f'Cascade: HeadConcept top-{TOP_K} heads per layer (deep → shallow)', fontsize=10)
plt.tight_layout()
plt.show()

print('selected heads per layer (deep → shallow):')
for layer in cascade_layer_names:
    print(f'  {layer}  →  H{selected[layer]}')

## 8. Notes & next steps

* **Magnitude regime.** With AlphaBeta(0.5, 0.5) the input |R|_max
  is O(10²) and conservation is within a few × the target logit
  (vs ~10²² magnitudes under the standard `2y+ε` rule — see
  `RESEARCH_NOTES.md` Entry 6).
* **One concept, many sites.** `HeadConcept` and `EmbeddingDimConcept`
  are shape-agnostic across the LRP inspection sites — the SAME
  concept instance can be hooked at `q_lrp_probe`, `k_lrp_probe`,
  `v_lrp_probe`, or `proj_drop` and aggregates per-head /
  per-embedding-dim correctly because all four sites carry the
  same `(B, N, embed_dim)` shape. Use the same instance across
  multiple layer entries in `fv_specs` to compare interpretations.
* **Token-role masking.** All three concepts accept `token_filter=slice(...)`.
  Default = include all tokens. For spatial-only attribution pass
  `token_filter=slice(NUM_PREFIX_TOKENS, None)`; for prefix-only,
  `slice(0, NUM_PREFIX_TOKENS)`.
* **Skip the LAST block for prefix-token attribution.** The
  classification head reads only the cls token, so register-token
  outputs at the last block's `proj_drop` have a strict mathematical
  zero of LRP relevance (no path to the logit). Use an earlier block.
* **FV indexing scope.** Default = whole dataset. For fast iteration
  during development, swap `len(dataset)` → a small integer in
  `fv.run(...)`. Cache lives at `data/fv_cache/<base>_<head>_<train-ds>/<tag>`
  and is reused on re-runs (skip-if-indexed check covers all layers).
* **`AttnWeightConcept` is intentionally absent.** Softmax weights
  have no fixed semantic per neuron (the same cell combines
  different concepts for different inputs), so reference-sample
  retrieval would be uninformative. The `attn.softmax` submodule
  is still hookable for direct attention-map inspection via
  `record_layer=['backbone.blocks.{i}.attn.softmax']` — useful for K/Q
  relation analysis but not for concept identification.